# Qwen3.8-27B OCR-VQA — 9시간 자동 후보 검증 + 전체 데이터 최종 학습

이 노트북은 기존 0.95918 실험 설정을 기준으로 **9시간 이내 종료**를 목표로 한다.

- 후보 실험: 기존 설정에서 증강 Random View 확률만 70% → 80%로 변경, 2 Epoch
- 자동 선택: 현재 기준 95.1111%와 비교해 후보가 실질적으로 좋을 때만 80% 채택
- 최종 학습: 선택 설정으로 Train 6,714장 전체 사용
- 학습 강도 고정: 총 1,510 optimizer step에서 종료(기존 성공 실험과 거의 동일)
- 시간 보호: 학습 시작 후 7.5시간에 최종 학습을 안전 중단하고 추론 시간 확보
- 제출: 다른 제출 CSV를 섞지 않고 최종 단일 모델의 직접 추론만 저장

> 반드시 위에서 아래로 순서대로 실행한다. 기존 체크포인트와 노트북은 덮어쓰지 않는다.


# 실행 개요

1. G4 GPU·패키지·Drive·원본/증강 데이터 확인
2. Qwen3.8-27B BF16 + DeltaNet LoRA 로드
3. 기존 group-aware split에서 증강 확률 80% 후보를 2 Epoch 검증
4. 현재 기준과 후보 결과 자동 비교
5. LoRA를 초기화하고 전체 6,714장으로 최종 학습
6. 총 1,510 optimizer step 또는 7.5시간 보호선에서 종료
7. 최종 단일 모델로 Test 추론 및 `submission_final_single_model.csv` 생성

예상 시간은 후보 학습 약 3시간, 전체 데이터 최종 학습 약 3시간, 추론·저장 포함 약 7~8시간이다.


# 환경 준비

Colab 런타임 유형을 **G4 GPU**로 선택한 뒤 실행하세요.

- torch는 **재설치하지 않습니다.** (기존의 `--index-url .../cu121` 재설치는 Blackwell(sm_120)에서 동작하지 않습니다.)
- `transformers>=5` 필요 (Qwen3.5 계열)
- `flash-linear-attention`: Qwen3.5의 Gated DeltaNet 레이어 fast path용. 없으면 느린 PyTorch fallback으로 동작합니다.

In [ ]:
# [27B-SMOKE] Colab G4(Blackwell, sm_120) + Qwen3.5용 패키지
# - torch/torchvision은 재설치하지 않는다 (Colab 기본 torch 사용).
# - 기존 노트북의 `pip install torchao>=0.16.0` 은 쉘에서 `>` 가 리다이렉션으로 해석되므로 따옴표로 감싼다.
# - pandas / pillow 버전 pin은 Colab 기본 패키지와 충돌할 수 있어 제거했다.
!pip -q install -U "transformers>=5" accelerate peft datasets scikit-learn
!pip -q install -U bitsandbytes "torchao>=0.16.0"
!pip -q install -U flash-linear-attention

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)

try:
    import causal_conv1d
    print("causal_conv1d: 설치됨")
except ImportError:
    print("causal_conv1d: 없음")

try:
    import fla
    print("fla: 설치됨")
except ImportError:
    print("fla: 없음")

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Transformers: 5.17.0
causal_conv1d: 없음


fla: 설치됨


In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
!pip install -U git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation

In [ ]:
import causal_conv1d

print(causal_conv1d.__version__)

1.7.0


In [ ]:
!pip install -U wandb

In [ ]:
import causal_conv1d
import wandb

print("causal_conv1d:", causal_conv1d.__version__)
print("wandb:", wandb.__version__)

causal_conv1d: 1.7.0
wandb: 0.30.0


In [ ]:
import torch
import transformers
import causal_conv1d
import wandb

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("causal_conv1d:", causal_conv1d.__version__)
print("wandb:", wandb.__version__)

PyTorch: 2.11.0+cu128
CUDA: 12.8
Transformers: 5.17.0
causal_conv1d: 1.7.0
wandb: 0.30.0


In [ ]:
!wandb login

In [ ]:
# [27B-SMOKE] 환경 점검
import os
# 메모리 단편화 완화 (torch import 전에 설정해야 효과가 있다. 이미 import 했다면 런타임 재시작 후 실행)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import shutil, importlib.metadata as md
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 G4 GPU로 변경하세요."
cap = torch.cuda.get_device_capability(0)
arch_list = torch.cuda.get_arch_list()
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):", round(props.total_memory / 1024**3, 2))
print("Compute capability:", cap)
print("torch arch list:", arch_list)

if cap[0] >= 12 and not any(a in arch_list for a in ("sm_120", "compute_120")):
    print("[FAIL] 이 torch 빌드는 sm_120(Blackwell)을 지원하지 않습니다. cu128 이상 torch가 필요합니다.")
if props.total_memory / 1024**3 < 80:
    print("[WARN] VRAM이 80GB 미만입니다. 27B bf16 LoRA는 어렵습니다. QUANT_MODE='4bit' 를 고려하세요.")

def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return "NOT INSTALLED"

for pkg in ["transformers", "peft", "accelerate", "bitsandbytes", "torchao", "flash-linear-attention", "causal-conv1d"]:
    print(f"{pkg:26s}", _ver(pkg))

tf_major = int(_ver("transformers").split(".")[0]) if _ver("transformers")[0].isdigit() else 0
if tf_major < 5:
    print("[FAIL] transformers v5 이상이 필요합니다. 설치 셀 실행 후 런타임을 다시 시작하세요.")
if _ver("flash-linear-attention") == "NOT INSTALLED":
    print("[WARN] flash-linear-attention 미설치 → Gated DeltaNet이 느린 fallback으로 동작합니다.")

try:
    import psutil
    print("System RAM(GB):", round(psutil.virtual_memory().total / 1024**3, 1))
except Exception:
    pass
free_gb = shutil.disk_usage("/").free / 1024**3
print("Disk free(GB):", round(free_gb, 1))
if free_gb < 90:
    print("[WARN] 디스크 여유가 90GB 미만입니다. 27B 가중치(약 56GB) 다운로드 + 데이터 압축 해제 공간이 부족할 수 있습니다.")

Torch version: 2.11.0+cu128
CUDA version: 12.8
cuDNN version: 91900
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM(GB): 94.97
Compute capability: (12, 0)
torch arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
transformers               5.17.0
peft                       0.21.0
accelerate                 1.15.0
bitsandbytes               0.50.2
torchao                    0.18.0
flash-linear-attention     0.5.2
causal-conv1d              1.7.0
System RAM(GB): 176.9
Disk free(GB): 187.5


# 데이터 준비

Google Drive에서 다음 두 압축 파일을 `/content`에 자동으로 해제합니다.

1. `ssafy-16-2-ai.zip`: 원본 `train.csv`, `test.csv`, 이미지 폴더
2. `vqa_augmented_manifest.zip`: 증강 이미지, `train_all_augmented.csv`, lineage manifest

증강 ZIP이 아직 없더라도 이미 `/content/train_all_augmented.csv`가 존재하면 그대로 사용합니다. 둘 다 없으면 이후 `USE_AUGMENTED_DATA=True` 설정에서 명확한 오류를 출력합니다.


In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 원본 데이터 + 증강 데이터 자동 압축 해제
# ============================================================
import os
import subprocess

ORIGINAL_ZIP_PATH = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ORIGINAL_ZIP_PASSWORD = os.environ.get("VQA_ZIP_PASSWORD", "")  # 대회 제공 비밀번호는 환경 변수로 전달
AUGMENTED_ZIP_PATH = "/content/drive/MyDrive/vqa_augmented_manifest.zip"
EXTRACT_ROOT = "/content"


def extract_7z(zip_path: str, password: str | None = None) -> None:
    command = ["7z", "x", "-y", f"-o{EXTRACT_ROOT}"]
    if password:
        command.append(f"-p{password}")
    command.append(zip_path)
    print("Extracting:", zip_path)
    subprocess.run(command, check=True)


if not os.path.exists(ORIGINAL_ZIP_PATH):
    raise FileNotFoundError(f"원본 데이터 ZIP이 없습니다: {ORIGINAL_ZIP_PATH}")
extract_7z(ORIGINAL_ZIP_PATH, ORIGINAL_ZIP_PASSWORD)

if os.path.exists(AUGMENTED_ZIP_PATH):
    extract_7z(AUGMENTED_ZIP_PATH)
elif os.path.exists("/content/train_all_augmented.csv"):
    print("증강 CSV가 이미 /content에 있어 기존 파일을 사용합니다.")
else:
    print(
        "[WARN] 증강 ZIP/CSV가 없습니다. 먼저 증강 노트북을 실행하거나 "
        "학습 설정에서 USE_AUGMENTED_DATA=False로 변경하세요."
    )

required_original = [
    "/content/train.csv", "/content/test.csv", "/content/train", "/content/test"
]
missing_original = [path for path in required_original if not os.path.exists(path)]
if missing_original:
    raise FileNotFoundError(f"원본 압축 해제 결과 누락: {missing_original}")

if os.path.exists("/content/train_all_augmented.csv"):
    augmented_requirements = [
        "/content/train_rotate_aug",
        "/content/train_quality_aug",
        "/content/train_text_crop",
    ]
    missing_augmented = [path for path in augmented_requirements if not os.path.exists(path)]
    if missing_augmented:
        raise FileNotFoundError(f"증강 압축 해제 결과 누락: {missing_augmented}")
    print("원본 및 증강 데이터 준비 완료")
else:
    print("원본 데이터 준비 완료 — 증강 데이터는 아직 준비되지 않았습니다.")


Extracting: /content/drive/MyDrive/ssafy-16-2-ai.zip
Extracting: /content/drive/MyDrive/vqa_augmented_manifest.zip
원본 및 증강 데이터 준비 완료


# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random, json, shutil, time, gc, unicodedata
from contextlib import nullcontext
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader

import torch
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoProcessor, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU 런타임이 필요합니다."
PIPELINE_STARTED_AT = time.time()

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("BF16 supported:", torch.cuda.is_bf16_supported())

# ============================================================
# 실험 설정 — 96GB G4에서 실사용 90GB 이하 목표
# ============================================================
MODEL_ID = "Qwen/Qwen3.8-27B"
QUANT_MODE = "bf16"

# OCR 우선 해상도. probe가 88GB 목표를 넘으면 순서대로 자동 하향한다.
MIN_PIXELS = 384 * 384
PIXEL_CANDIDATES = [896 * 896, 832 * 832, 768 * 768, 672 * 672, 576 * 576]
MAX_PIXELS = PIXEL_CANDIDATES[0]
IMAGE_SIZE = int(math.sqrt(MAX_PIXELS))

MAX_VRAM_GB = 90.0
TARGET_MAX_VRAM_GB = 88.0
OPTIMIZER_SAFETY_MARGIN_GB = 1.5
RUN_ADAPTIVE_MEMORY_PROBE = True

MAX_NEW_TOKENS = 2
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_SUFFIXES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
# Qwen3.8의 48개 DeltaNet 계층 projection을 함께 학습한다.
EXTRA_LORA_SUFFIXES = [
    "in_proj_a", "in_proj_b", "in_proj_qkv", "in_proj_z", "out_proj",
]

VAL_RATIO = 0.10
SEED = 42
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 1
EARLY_STOPPING_MIN_DELTA = 1e-4

RUN_BASELINE_VALIDATION = False
CONSTRAINED_VERIFY_SAMPLES = 64
CONSTRAINED_MIN_AGREEMENT = 0.98
VALIDATION_MODE = "auto"
INFERENCE_MODE = "auto"

SMOKE_TEST = False
SMOKE_TRAIN_SAMPLES = 200
SMOKE_MAX_TRAIN_BATCHES = 32
SMOKE_VAL_ACC_SAMPLES = 32
SMOKE_INFER_SAMPLES = 16
LIMITED_TRAIN_SAMPLES: Optional[int] = None

RESUME_CHECKPOINT = None
CHECKPOINT_ROOT = "/content/drive/MyDrive/vqa_checkpoints"
EXPERIMENT_NAME = "qwen38_27b_ocr_90gb_aug80_candidate_9h"
RUN_EXPERIMENT_NAME = EXPERIMENT_NAME + ("_smoke" if SMOKE_TEST else "")
EXPERIMENT_DIR = os.path.join(CHECKPOINT_ROOT, RUN_EXPERIMENT_NAME)
SAVE_EVERY_EPOCH = True
SAVE_TOTAL_LIMIT = 2
USE_WANDB = True

# ============================================================
# 선택적 증강 데이터 사용
# - random_view(권장): 원본별로 매 Epoch 한 view만 사용하여 학습 시간 증가 없음
# - expand: 선택한 모든 증강 행을 실제 학습 행으로 추가 (매우 느림)
# ============================================================
USE_AUGMENTED_DATA = True
AUGMENTED_CSV = "/content/train_all_augmented.csv"
AUGMENTATION_TRAIN_MODE = "random_view"  # "random_view" | "expand"
AUGMENTED_VIEW_PROB = 0.80
ALLOWED_AUGMENTATIONS = [
    "rot_p5", "rot_m5", "rot_p10", "rot_m10", "perspective",
    "downscale", "blur", "jpeg", "textcrop",
]
# expand 모드에서 원본당 사용할 최대 증강 수. None이면 모두 사용한다.
MAX_AUGMENTATIONS_PER_SOURCE = 2

# ============================================================
# 9시간 자동 최종 학습 설정
# 현재 성공 실험: group-aware validation 675개 기준 642개 정답
# 후보가 2문항 이상 개선하거나, 동일 accuracy에서 loss가 낮을 때만 80%를 채택한다.
# ============================================================
BASELINE_VALID_ACCURACY = 0.951111
BASELINE_VALID_LOSS = 0.0534
CANDIDATE_AUGMENTED_VIEW_PROB = 0.80
BASELINE_AUGMENTED_VIEW_PROB = 0.70
FINAL_MAX_OPTIMIZER_STEPS = 1510
FINAL_MAX_EPOCHS = 2
FINAL_TRAINING_STOP_HOURS = 7.5
FINAL_EXPERIMENT_NAME = "qwen38_27b_ocr_90gb_full6714_autofinal_9h"
FINAL_EXPERIMENT_DIR = os.path.join(CHECKPOINT_ROOT, FINAL_EXPERIMENT_NAME)


REPORT: Dict[str, Any] = {
    "model_id": MODEL_ID,
    "quant_mode": QUANT_MODE,
    "vram_budget_gb": MAX_VRAM_GB,
}
# 기존 모델 로드 셀과의 호환용 별칭
SMOKE_REPORT = REPORT

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
N_TRAIN_FULL = len(train_df)
N_TEST = len(test_df)

def absolute_content_path(value: str) -> str:
    value = str(value)
    return value if os.path.isabs(value) else os.path.join("/content", value)

train_df["path"] = train_df["path"].map(absolute_content_path)
test_df["path"] = test_df["path"].map(absolute_content_path)

train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
valid_answers = {"a", "b", "c", "d"}
invalid_answers = sorted(set(train_df["answer"]) - valid_answers)
if invalid_answers:
    raise ValueError(f"Unexpected answer labels: {invalid_answers}")

if SMOKE_TEST:
    per_class = max(2, SMOKE_TRAIN_SAMPLES // 4)
    parts = [
        group.sample(min(per_class, len(group)), random_state=SEED + idx)
        for idx, (_, group) in enumerate(train_df.groupby("answer"))
    ]
    train_df = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
elif LIMITED_TRAIN_SAMPLES is not None:
    # group split 이전의 제한 실험용. 클래스별 동일 개수를 뽑는다.
    per_class = LIMITED_TRAIN_SAMPLES // 4
    parts = [
        group.sample(min(per_class, len(group)), random_state=SEED + idx)
        for idx, (_, group) in enumerate(train_df.groupby("answer"))
    ]
    train_df = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
else:
    train_df = train_df.reset_index(drop=True)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
print("Experiment dir:", EXPERIMENT_DIR)
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Initial pixel budget:", MIN_PIXELS, "~", MAX_PIXELS)
print("Effective batch:", PER_DEVICE_BATCH_SIZE * GRAD_ACCUM)


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM(GB): 94.97
BF16 supported: True
Experiment dir: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_aug80_candidate_9h
Train rows: 6714 | Test rows: 6714
Initial pixel budget: 147456 ~ 802816
Effective batch: 8


# 모델, Processor

기본 모델은 `Qwen/Qwen3.5-27B` 입니다.

- `QUANT_MODE="bf16"`: 16-bit LoRA (가중치 약 56GB, G4 96GB 권장 경로)
- `QUANT_MODE="4bit"`: NF4 QLoRA fallback
- `device_map={"": 0}` : **GPU 0에만 올립니다.** `"auto"`는 GPU에 안 들어가면 CPU로 offload 해서 "올라갔지만 매우 느린" 상태가 되어 smoke test 결과를 왜곡할 수 있습니다.
- LoRA 대상은 이름이 아닌 **실제 모듈 경로 목록**으로 지정하고 vision tower는 제외합니다.

In [ ]:
# Qwen3.8-27B 로드 — bf16 LoRA / DeltaNet 포함
from collections import Counter

try:
    from transformers import AutoModelForMultimodalLM as VLMAutoModel
except ImportError:
    from transformers import AutoModelForImageTextToText as VLMAutoModel
print("Model class:", VLMAutoModel.__name__)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# ---------------- processor ----------------
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
except Exception as e:
    print("[WARN] min/max_pixels 인자로 processor 생성 실패 → 기본값으로 생성 후 수동 설정:", repr(e)[:200])
    processor = AutoProcessor.from_pretrained(MODEL_ID)

# Qwen3.5 image processor가 min/max_pixels 인자를 무시할 수 있으므로 값을 직접 확인/설정한다.
ip = processor.image_processor
_size = getattr(ip, "size", None)
try:
    if isinstance(_size, dict):
        ip.size = {"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS}
    elif _size is not None:  # SizeDict 등 속성 접근형
        _size.shortest_edge = MIN_PIXELS
        _size.longest_edge = MAX_PIXELS
except Exception as e:
    print("[WARN] image_processor.size 수동 설정 실패:", repr(e)[:200])
for attr, val in (("min_pixels", MIN_PIXELS), ("max_pixels", MAX_PIXELS)):
    if hasattr(ip, attr):
        try:
            setattr(ip, attr, val)
        except Exception:
            pass
print("image_processor:", type(ip).__name__,
      "| size:", getattr(ip, "size", None),
      "| patch_size:", getattr(ip, "patch_size", None),
      "| merge_size:", getattr(ip, "merge_size", None))

# ---------------- model ----------------
load_kwargs = dict(device_map={"": 0}, attn_implementation="sdpa")
if QUANT_MODE == "4bit":
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
elif QUANT_MODE != "bf16":
    raise ValueError("QUANT_MODE must be 'bf16' or '4bit'")

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
try:
    base_model = VLMAutoModel.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **load_kwargs)
except TypeError:  # 구버전 인자명
    base_model = VLMAutoModel.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **load_kwargs)
load_sec = time.time() - t0

n_params = sum(p.numel() for p in base_model.parameters())
mem_after_load = torch.cuda.memory_allocated() / 1024**3
print(f"Loaded in {load_sec:.0f}s | params: {n_params/1e9:.2f}B | GPU mem allocated: {mem_after_load:.1f} GB")
SMOKE_REPORT.update(load_sec=load_sec, params_b=n_params / 1e9, mem_after_load_gb=mem_after_load)

# ---------------- 학습 준비 ----------------
if QUANT_MODE == "4bit":
    base_model = prepare_model_for_kbit_training(base_model)

for _cfg in (base_model.config, getattr(base_model.config, "text_config", None)):
    if _cfg is not None and hasattr(_cfg, "use_cache"):
        _cfg.use_cache = False

if hasattr(base_model, "enable_input_require_grads"):
    base_model.enable_input_require_grads()

base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# ---------------- LoRA 대상 모듈 탐색 ----------------
def _is_vision(name: str) -> bool:
    return ("visual" in name) or ("vision" in name)

leaf_counter = Counter()
for n, m in base_model.named_modules():
    if isinstance(m, torch.nn.Linear):
        leaf_counter[("vision" if _is_vision(n) else "language", n.split(".")[-1])] += 1

print("\n[Linear 모듈 이름 분포 — (영역, 이름): 개수]")
for k, v in sorted(leaf_counter.items()):
    print(f"  {k}: {v}")

lora_suffixes = set(LORA_TARGET_SUFFIXES) | set(EXTRA_LORA_SUFFIXES)
lora_target_names = [
    n for n, m in base_model.named_modules()
    if isinstance(m, torch.nn.Linear)
    and n.split(".")[-1] in lora_suffixes
    and not _is_vision(n)
]
assert len(lora_target_names) > 0, "LoRA 대상 모듈을 찾지 못했습니다. 위 Linear 이름 분포를 보고 접미사를 수정하세요."
print(f"\nLoRA target modules: {len(lora_target_names)}개 (vision 제외)")

# ============================================================
# checkpoint resume 여부에 따라 LoRA adapter를 생성/로드한다.
# 서로 다른 hyperparameter 실험은 checkpoint를 공유하지 않는다.
# ============================================================
if RESUME_CHECKPOINT is not None:
    adapter_config_path = os.path.join(RESUME_CHECKPOINT, "adapter_config.json")
    if not os.path.exists(adapter_config_path):
        raise FileNotFoundError(
            f"RESUME_CHECKPOINT에 adapter_config.json이 없습니다: {RESUME_CHECKPOINT}"
        )

    model = PeftModel.from_pretrained(
        base_model,
        RESUME_CHECKPOINT,
        is_trainable=True,
    )
    print("Loaded LoRA adapter from:", RESUME_CHECKPOINT)
else:
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=lora_target_names,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

# device_map={"": 0} 로 단일 GPU에 배치했으므로 model.to(device)는 호출하지 않는다.
MODEL_DEVICE = next(model.parameters()).device
print("Model device:", MODEL_DEVICE)

mem_after_lora = torch.cuda.memory_allocated() / 1024**3
print(f"GPU mem allocated after LoRA: {mem_after_lora:.1f} GB")
SMOKE_REPORT.update(mem_after_lora_gb=mem_after_lora, n_lora_modules=len(lora_target_names))

def set_processor_pixel_budget(min_pixels: int, max_pixels: int) -> None:
    """Qwen image processor의 면적 기반 pixel budget을 안전하게 갱신한다."""
    global MIN_PIXELS, MAX_PIXELS, IMAGE_SIZE
    MIN_PIXELS = int(min_pixels)
    MAX_PIXELS = int(max_pixels)
    IMAGE_SIZE = int(math.sqrt(MAX_PIXELS))
    image_processor = processor.image_processor

    for attr, value in (("min_pixels", MIN_PIXELS), ("max_pixels", MAX_PIXELS)):
        if hasattr(image_processor, attr):
            try:
                setattr(image_processor, attr, value)
            except Exception:
                pass

    size = getattr(image_processor, "size", None)
    if isinstance(size, dict):
        image_processor.size = {"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS}
    elif size is not None:
        if hasattr(size, "shortest_edge"):
            size.shortest_edge = MIN_PIXELS
        if hasattr(size, "longest_edge"):
            size.longest_edge = MAX_PIXELS

    print(f"Processor pixel budget → min={MIN_PIXELS:,}, max={MAX_PIXELS:,} (~{IMAGE_SIZE}px square)")

set_processor_pixel_budget(MIN_PIXELS, MAX_PIXELS)


Model class: AutoModelForMultimodalLM


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/8.95k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

... (7 outputs left out)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

Loaded in 167s | params: 27.36B | GPU mem allocated: 51.0 GB

[Linear 모듈 이름 분포 — (영역, 이름): 개수]
  ('language', 'down_proj'): 64
  ('language', 'gate_proj'): 64
  ('language', 'in_proj_a'): 48
  ('language', 'in_proj_b'): 48
  ('language', 'in_proj_qkv'): 48
  ('language', 'in_proj_z'): 48
  ('language', 'k_proj'): 16
  ('language', 'lm_head'): 1
  ('language', 'o_proj'): 16
  ('language', 'out_proj'): 48
  ('language', 'q_proj'): 16
  ('language', 'up_proj'): 64
  ('language', 'v_proj'): 16
  ('vision', 'linear_fc1'): 28
  ('vision', 'linear_fc2'): 28
  ('vision', 'proj'): 27
  ('vision', 'qkv'): 27

LoRA target modules: 496개 (vision 제외)
trainable params: 116,727,808 || all params: 27,473,456,368 || trainable%: 0.4249
Model device: cuda:0
GPU mem allocated after LoRA: 51.4 GB
Processor pixel budget → min=147,456, max=802,816 (~896px square)


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

> **[27B-SMOKE] 중요**: Qwen3.5는 **thinking이 기본 ON** 입니다. 학습/검증/추론 모든 곳의
> `apply_chat_template`에 `enable_thinking=False`를 동일하게 전달해야 합니다.
> (그렇지 않으면 generation prompt가 `<think>\n`으로 끝나 `max_new_tokens=2`로는 답을 내기 전에 끊깁니다.)
> 아래 `render_chat()` 한 함수로 통일했습니다.

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert Visual Question Answering system specialized in multilingual OCR "
    "(Korean, English, Chinese, Japanese) and complex spatial visual reasoning.\n"
    "When answering:\n"
    "1. Accurately transcribe multilingual text, numbers, symbols, and layout structure from the image.\n"
    "2. Analyze spatial relationships (e.g., floor numbers, top/bottom, left/right, row/column alignments).\n"
    "3. Compare the visual details against all options. If no option matches 100% perfectly or if the query is ambiguous, select the single most plausible and closest option.\n"
    "4. Output ONLY one lower-case letter among a, b, c, or d. Do not output any explanation or extra text."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "[분석 가이드]\n"
        "1. 이미지의 텍스트(한글, 영어, 한자/중국어, 일어, 숫자) 및 위치/층수/방향 관계를 정밀하게 확인하세요.\n"
        "2. 건물 안내도, 표지판, 표 등의 구조와 질문의 조건이 일치하는지 각 보기(a,b,c,d)를 세밀히 대조하세요.\n"
        "3. 만약 완벽히 일치하는 보기가 없거나 헷갈리는 경우, 이미지 맥락상 가장 가깝고 타당한 보기 하나를 선택하세요.\n\n"
        f"질문: {question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# [Prompt Engineering 개선 경과]
# 1. 역할 & 포맷 고정: System Prompt(영어)를 활용해 '소문자 한 글자(a~d)' strict 출력 포맷 준수율 확보
# 2. 다국어 OCR 강화: 한/영/중/일 텍스트, 숫자, 기호 등 도메인 특성(DocVQA/TextVQA)에 맞춘 정밀 판독 유도
# 3. 공간/레이아웃 추론: 층별 안내도, 표지판 등 위치/방향/구조(Spatial Reasoning) 관계 분석 지침 추가
# 4. 예외/모호성 대응: 선지 오타/모호함 대비 맥락상 가장 타당한 답을 고르는 'Closest Match Rule' 적용
# 5. 기대 효과: 포맷 이탈율 방지 + 고난도 visual text/공간 결합 문항의 변별력 및 Logprob 추론 승률 극대화

# ------------------------------------------------------------
# [27B-SMOKE] chat template 렌더링을 한 곳으로 통일한다.
# Qwen3.5는 thinking 기본 ON → 학습/검증/추론 모두 enable_thinking=False 로 고정.
# (템플릿이 모르는 kwarg는 jinja 변수로만 전달되므로 다른 모델에서도 오류는 나지 않는다.)
# ------------------------------------------------------------
CHAT_KWARGS = dict(enable_thinking=False)

def render_chat(processor, messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        **CHAT_KWARGS,
    )

# 학습/검증/추론에서 동일한 메시지 구조를 재사용한다.
def build_messages(question, a, b, c, d, image, answer=None):
    user_text = build_mc_prompt(question, a, b, c, d)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text},
        ]},
    ]

    if answer is not None:
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": str(answer).strip().lower()}]}
        )

    return messages

# 출력에서 a/b/c/d만 추출한다.
# 잘못된 출력은 None으로 유지하여 validation에서 오답으로 처리한다.
def extract_choice(text: str) -> Optional[str]:
    text = str(text).strip().lower()
    matches = re.findall(r"\b([abcd])\b", text)
    return matches[-1] if matches else None

# Custom Dataset, Collator

학습 collator에서는 **label masking**을 적용합니다.

정답 이전의 system/user/image prompt token은 `-100`으로 masking하고,
assistant의 정답 영역만 loss 계산 대상으로 둡니다.

Validation generation에서는 정답을 prompt에 넣지 않습니다.

> **[27B-SMOKE]** prompt 길이를 `attention_mask.sum()`으로 계산하므로 **right padding**이 전제입니다.
> Qwen 계열 tokenizer의 기본 padding side가 모델마다 다를 수 있어 collator에서 명시적으로 right로 고정했습니다.

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages = build_messages(q, a, b, c, d, img, answer=gold)
        else:
            # validation generation에서는 정답을 절대 prompt에 넣지 않는다.
            messages = build_messages(q, a, b, c, d, img, answer=None)

        return {
            "messages": messages,
            "image": img,
            "answer": str(row["answer"]).strip().lower(),
        }


# ============================================================
# Label masking이 적용된 DataCollator
# - system/user/image 부분은 -100으로 masking한다.
# - assistant의 정답 토큰(+ EOS 등 답변 이후 토큰)만 loss 대상이 된다.
# ============================================================
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __post_init__(self):
        # [27B-SMOKE] prompt_len 계산이 right padding을 전제로 하므로 명시적으로 고정한다.
        if self.train:
            self.processor.tokenizer.padding_side = "right"

    def __call__(self, batch):
        texts_full = []
        texts_prompt = []
        images = []

        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            # 정답까지 포함된 full conversation
            texts_full.append(render_chat(self.processor, messages, add_generation_prompt=False))
            images.append(img)

            if self.train:
                # 같은 입력에서 assistant 답변 직전까지만 구성한다.
                texts_prompt.append(render_chat(self.processor, messages[:-1], add_generation_prompt=True))

        # full conversation 인코딩
        enc = self.processor(
            text=texts_full,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if not self.train:
            return enc

        # answer 이전 prompt 길이를 계산하기 위한 별도 인코딩
        prompt_enc = self.processor(
            text=texts_prompt,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"]
        labels = input_ids.clone()

        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        # 각 샘플의 prompt 구간 전체를 loss에서 제외한다.
        for row_idx in range(input_ids.shape[0]):
            prompt_len = int((prompt_enc["attention_mask"][row_idx] == 1).sum().item())
            labels[row_idx, :prompt_len] = -100

        enc["labels"] = labels
        return enc

class VQAMultiViewDataset(Dataset):
    """원본 1개당 원본 또는 사전 생성된 증강 view 하나를 결정적으로 선택한다."""
    def __init__(self, base_df, variants_df, processor, augmented_probability=0.7, seed=42):
        self.base_df = base_df.reset_index(drop=True).copy()
        self.processor = processor
        self.augmented_probability = float(augmented_probability)
        self.seed = int(seed)
        self.epoch = 0
        self.variant_groups = {
            source_id: group.reset_index(drop=True)
            for source_id, group in variants_df.groupby("source_id", sort=False)
        }

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.base_df)

    def __getitem__(self, index):
        original = self.base_df.iloc[index]
        source_id = str(original["id"])
        variants = self.variant_groups.get(source_id)

        # index와 epoch 기반 RNG로 재실행 가능한 view 선택
        rng = random.Random(self.seed + self.epoch * 1_000_003 + index)
        use_augmented = (
            variants is not None
            and len(variants) > 0
            and rng.random() < self.augmented_probability
        )
        row = variants.iloc[rng.randrange(len(variants))] if use_augmented else original

        image = Image.open(row["path"]).convert("RGB")
        question = str(row["question"])
        options = [str(row[key]) for key in "abcd"]
        answer = str(row["answer"]).strip().lower()
        messages = build_messages(question, *options, image, answer=answer)
        return {
            "messages": messages,
            "image": image,
            "answer": answer,
            "source_id": source_id,
            "augmentation": str(row.get("augmentation", "original")),
        }


In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(
        entity="ssafy16-b062",
        project="ssafy16-aichall",
        name=RUN_EXPERIMENT_NAME,
        config={
            "model": MODEL_ID,
            "min_pixels": MIN_PIXELS,
            "max_pixels_initial": MAX_PIXELS,
            "vram_budget_gb": MAX_VRAM_GB,
            "epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "micro_batch": PER_DEVICE_BATCH_SIZE,
            "grad_accum": GRAD_ACCUM,
            "effective_batch": PER_DEVICE_BATCH_SIZE * GRAD_ACCUM,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_targets": sorted(set(LORA_TARGET_SUFFIXES + EXTRA_LORA_SUFFIXES)),
            "val_ratio": VAL_RATIO,
            "seed": SEED,
        },
    )
else:
    print("Weights & Biases logging disabled")


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# ============================================================
# Group-aware stratified split
# 동일한 정규화 질문 문구는 train/validation 양쪽에 나뉘지 않는다.
# ============================================================
def normalize_question(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

groups = train_df["question"].map(normalize_question)
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_idx, valid_idx = next(splitter.split(train_df, train_df["answer"], groups=groups))

train_subset = train_df.iloc[train_idx].reset_index(drop=True)
valid_subset = train_df.iloc[valid_idx].reset_index(drop=True)

train_groups = set(train_subset["question"].map(normalize_question))
valid_groups = set(valid_subset["question"].map(normalize_question))
assert train_groups.isdisjoint(valid_groups), "질문 group leakage가 발견되었습니다."

print("Train samples:", len(train_subset))
print("Valid samples:", len(valid_subset))
print("Question-group overlap:", len(train_groups & valid_groups))
print("Train distribution:", train_subset["answer"].value_counts(normalize=True).sort_index().round(4).to_dict())
print("Valid distribution:", valid_subset["answer"].value_counts(normalize=True).sort_index().round(4).to_dict())

train_subset.to_csv(os.path.join(EXPERIMENT_DIR, "train_split.csv"), index=False)
valid_subset.to_csv(os.path.join(EXPERIMENT_DIR, "valid_split.csv"), index=False)

# Validation은 항상 원본 이미지만 사용한다.
augmented_summary = {"enabled": False, "mode": None, "rows": 0}
if USE_AUGMENTED_DATA:
    if not os.path.exists(AUGMENTED_CSV):
        raise FileNotFoundError(
            f"증강 CSV가 없습니다: {AUGMENTED_CSV}\n"
            "먼저 VQA_Augmentation_Manifest_Compatible.ipynb를 실행하고 결과 ZIP을 /content에 풀어주세요."
        )
    augmented_df = pd.read_csv(AUGMENTED_CSV)
    required_aug_columns = set(train_df.columns) | {"source_id", "augmentation", "is_augmented"}
    missing_columns = sorted(required_aug_columns - set(augmented_df.columns))
    if missing_columns:
        raise ValueError(f"증강 CSV 필수 컬럼 누락: {missing_columns}")

    augmented_df["source_id"] = augmented_df["source_id"].astype(str)
    augmented_df["augmentation"] = augmented_df["augmentation"].astype(str)
    augmented_df["path"] = augmented_df["path"].map(absolute_content_path)
    train_source_ids = set(train_subset["id"].astype(str))
    variants_df = augmented_df[
        augmented_df["is_augmented"].astype(bool)
        & augmented_df["source_id"].isin(train_source_ids)
        & augmented_df["augmentation"].isin(ALLOWED_AUGMENTATIONS)
    ].copy()

    # 증강 이미지가 validation source에서 파생되지 않았는지 강제 확인한다.
    valid_source_ids = set(valid_subset["id"].astype(str))
    assert set(variants_df["source_id"]).isdisjoint(valid_source_ids), "증강 train/validation leakage 발견"

    if AUGMENTATION_TRAIN_MODE == "random_view":
        train_ds = VQAMultiViewDataset(
            train_subset,
            variants_df,
            processor,
            augmented_probability=AUGMENTED_VIEW_PROB,
            seed=SEED,
        )
    elif AUGMENTATION_TRAIN_MODE == "expand":
        if MAX_AUGMENTATIONS_PER_SOURCE is not None:
            variants_df = (
                variants_df.groupby("source_id", group_keys=False)
                .sample(n=MAX_AUGMENTATIONS_PER_SOURCE, random_state=SEED, replace=False)
                if variants_df.groupby("source_id").size().min() >= MAX_AUGMENTATIONS_PER_SOURCE
                else variants_df.groupby("source_id", group_keys=False).head(MAX_AUGMENTATIONS_PER_SOURCE)
            )
        expanded_train = pd.concat([train_subset, variants_df[train_subset.columns]], ignore_index=True)
        train_ds = VQAMCDataset(expanded_train, processor, train=True)
    else:
        raise ValueError("AUGMENTATION_TRAIN_MODE must be 'random_view' or 'expand'")

    augmented_summary = {
        "enabled": True,
        "mode": AUGMENTATION_TRAIN_MODE,
        "variant_rows_available": len(variants_df),
        "sources_with_variants": int(variants_df["source_id"].nunique()),
        "train_dataset_length": len(train_ds),
        "view_probability": AUGMENTED_VIEW_PROB,
    }
    print("Augmentation:", augmented_summary)
else:
    train_ds = VQAMCDataset(train_subset, processor, train=True)

REPORT["augmentation"] = augmented_summary
valid_loss_ds = VQAMCDataset(valid_subset, processor, train=True)
valid_gen_ds = VQAMCDataset(valid_subset, processor, train=False)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_ds,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
valid_loss_loader = DataLoader(
    valid_loss_ds,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Effective batch size:", PER_DEVICE_BATCH_SIZE * GRAD_ACCUM)


Train samples: 6039
Valid samples: 675
Question-group overlap: 0
Train distribution: {'a': 0.2534, 'b': 0.2449, 'c': 0.2555, 'd': 0.2462}
Valid distribution: {'a': 0.2519, 'b': 0.2444, 'c': 0.2563, 'd': 0.2474}
Augmentation: {'enabled': True, 'mode': 'random_view', 'variant_rows_available': 54264, 'sources_with_variants': 6039, 'train_dataset_length': 6039, 'view_probability': 0.8}
Effective batch size: 8


# [27B-SMOKE] 사전 점검 ① 템플릿 / label masking ② 최악 조건 메모리 probe

학습을 돌리기 전에 아래를 확인합니다.

1. `full_text`가 `prompt_text`로 시작하는가 (학습/추론 프롬프트 일관성)
2. loss 대상 token이 **정답 글자(+종료 token)** 뿐인가 (label masking 정확성)
3. 최대 해상도 이미지 + 가장 긴 질문으로 batch 크기별 forward/backward를 돌려 peak VRAM과 OOM 지점을 확인

In [ ]:
# ============================================================
# 1) Prompt / label masking 검증
# ============================================================
tok = processor.tokenizer
items = [train_ds[i] for i in range(min(2, len(train_ds)))]
collator = DataCollator(processor, True)

full_text = render_chat(processor, items[0]["messages"], add_generation_prompt=False)
prompt_text = render_chat(processor, items[0]["messages"][:-1], add_generation_prompt=True)
prefix_ok = full_text.startswith(prompt_text)
batch_check = collator(items)

label_ok = True
for row_idx, item in enumerate(items):
    target = batch_check["labels"][row_idx]
    target = target[target != -100]
    decoded = tok.decode(target)
    print(f"[mask sample {row_idx}] gold={item['answer']} tokens={target.numel()} decoded={decoded!r}")
    label_ok &= 1 <= target.numel() <= 6 and decoded.strip().startswith(item["answer"])

print("template prefix OK:", prefix_ok)
print("label masking OK:", label_ok)
assert prefix_ok and label_ok, "Prompt prefix 또는 label masking 검증 실패"


# ============================================================
# 2) 90GB 제한 adaptive memory probe
# optimizer state는 아직 생성되지 않았으므로 AdamW FP32 moment(8 bytes/param)와
# 일시 버퍼 여유를 더해 예상 peak를 계산한다.
# ============================================================
def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()


def make_worst_case_item(pixel_budget: int):
    height = max(256, int(math.sqrt(pixel_budget / 3)))
    width = height * 3
    noise = Image.fromarray(np.random.randint(0, 255, (height, width, 3), dtype=np.uint8))
    lengths = (
        train_df["question"].astype(str).str.len()
        + train_df[["a", "b", "c", "d"]].astype(str).apply(lambda col: col.str.len()).sum(axis=1)
    )
    row = train_df.loc[lengths.idxmax()]
    answer = str(row["answer"]).strip().lower()
    messages = build_messages(row["question"], row["a"], row["b"], row["c"], row["d"], noise, answer=answer)
    return {"messages": messages, "image": noise, "answer": answer}


def probe_one_resolution(pixel_budget: int) -> Dict[str, Any]:
    set_processor_pixel_budget(MIN_PIXELS, pixel_budget)
    cleanup_cuda()
    item = make_worst_case_item(pixel_budget)
    probe_batch = DataCollator(processor, True)([item])
    probe_batch = {key: value.to(MODEL_DEVICE, non_blocking=True) for key, value in probe_batch.items()}

    model.train()
    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    started = time.time()
    with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE, enabled=True):
        output = model(**probe_batch)
        probe_loss = output.loss
    probe_loss.backward()
    torch.cuda.synchronize()

    peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
    trainable_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    estimated_adam = trainable_count * 8 / 1024**3
    estimated_total = peak_reserved + estimated_adam + OPTIMIZER_SAFETY_MARGIN_GB

    result = {
        "max_pixels": pixel_budget,
        "side": int(math.sqrt(pixel_budget)),
        "seq_len": int(probe_batch["input_ids"].shape[1]),
        "seconds": time.time() - started,
        "peak_alloc_gb": peak_alloc,
        "peak_reserved_gb": peak_reserved,
        "estimated_adam_gb": estimated_adam,
        "estimated_total_gb": estimated_total,
        "loss": float(probe_loss.item()),
    }
    model.zero_grad(set_to_none=True)
    del probe_batch, output, probe_loss
    cleanup_cuda()
    return result


probe_results = []
selected_pixels = None
if RUN_ADAPTIVE_MEMORY_PROBE:
    for candidate in PIXEL_CANDIDATES:
        try:
            result = probe_one_resolution(candidate)
            probe_results.append(result)
            print(
                f"[probe {result['side']}px] seq={result['seq_len']} "
                f"reserved={result['peak_reserved_gb']:.1f}GB "
                f"adam_est={result['estimated_adam_gb']:.1f}GB "
                f"total_est={result['estimated_total_gb']:.1f}GB"
            )
            if result["estimated_total_gb"] <= TARGET_MAX_VRAM_GB:
                selected_pixels = candidate
                break
        except torch.cuda.OutOfMemoryError:
            print(f"[probe {int(math.sqrt(candidate))}px] OOM → 다음 해상도로 하향")
            model.zero_grad(set_to_none=True)
            cleanup_cuda()
else:
    selected_pixels = MAX_PIXELS

if selected_pixels is None:
    raise RuntimeError(
        "576px에서도 예상 VRAM이 목표를 초과했습니다. "
        "LORA_R=8 또는 QUANT_MODE='4bit'로 낮추세요."
    )

set_processor_pixel_budget(MIN_PIXELS, selected_pixels)
REPORT["probe"] = probe_results
REPORT["selected_max_pixels"] = MAX_PIXELS
REPORT["selected_side"] = IMAGE_SIZE
print(f"선택된 최대 이미지 예산: {MAX_PIXELS:,} pixels (~{IMAGE_SIZE}px square)")


[mask sample 0] gold=a tokens=3 decoded='a<|im_end|>\n'
[mask sample 1] gold=a tokens=3 decoded='a<|im_end|>\n'
template prefix OK: True
label masking OK: True
Processor pixel budget → min=147,456, max=802,816 (~896px square)
[probe 896px] seq=1193 reserved=56.8GB adam_est=0.9GB total_est=59.1GB
Processor pixel budget → min=147,456, max=802,816 (~896px square)
선택된 최대 이미지 예산: 802,816 pixels (~896px square)


# Fine-tuning

`SMOKE_TEST=True`일 때는 **최대 `SMOKE_MAX_TRAIN_BATCHES` micro-batch만** 학습하고 epoch를 종료한 뒤,
validation loss / validation generation / checkpoint 저장까지 이어서 확인합니다.

- Model: Qwen3.5-27B
- 학습 방식: bf16 LoRA (`QUANT_MODE="bf16"`) 또는 4-bit QLoRA (`"4bit"`)
- LoRA: r=8, alpha=16, dropout=0.05 (vision tower 제외)
- Batch: micro-batch=2, gradient accumulation=1 → effective batch=2
- Epoch: 5 (validation accuracy가 개선되지 않으면 Early Stopping)
- Loss: answer token 중심
- Checkpoint: Google Drive (smoke 실행은 `..._smoke` 폴더에 저장)

`RESUME_CHECKPOINT`는 **같은 실험을 이어서 학습할 때만** 사용합니다.

In [ ]:
# ============================================================
# Training / validation utilities
# ============================================================
def build_choice_token_ids() -> Dict[str, int]:
    """실제 chat template 문맥에서 a/b/c/d의 첫 answer token ID를 추출한다."""
    sample = valid_gen_ds[0]
    messages = sample["messages"]
    prompt = render_chat(processor, messages, add_generation_prompt=True)
    prompt_ids = processor.tokenizer(prompt, add_special_tokens=False)["input_ids"]
    result = {}
    for choice in "abcd":
        base = messages[:]
        full_messages = base + [{"role": "assistant", "content": [{"type": "text", "text": choice}]}]
        full = render_chat(processor, full_messages, add_generation_prompt=False)
        full_ids = processor.tokenizer(full, add_special_tokens=False)["input_ids"]
        if full_ids[:len(prompt_ids)] != prompt_ids or len(full_ids) <= len(prompt_ids):
            raise RuntimeError(f"Choice token prefix mismatch: {choice}")
        result[choice] = full_ids[len(prompt_ids)]
    if len(set(result.values())) != 4:
        raise RuntimeError(f"Choice token IDs are not unique: {result}")
    return result


CHOICE_TOKEN_IDS = build_choice_token_ids()
print("Context-aware choice token IDs:", CHOICE_TOKEN_IDS)


@torch.no_grad()
def predict_choice_generate(messages, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    text = render_chat(processor, messages, add_generation_prompt=True)
    image = messages[1]["content"][0]["image"]
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(MODEL_DEVICE)
    generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_ids = generated[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
    return extract_choice(output_text), output_text, None


@torch.no_grad()
def predict_choice_constrained(messages):
    model.eval()
    text = render_chat(processor, messages, add_generation_prompt=True)
    image = messages[1]["content"][0]["image"]
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(MODEL_DEVICE)
    try:
        output = model(**inputs, logits_to_keep=1)
    except TypeError:
        output = model(**inputs)
    token_ids = [CHOICE_TOKEN_IDS[choice] for choice in "abcd"]
    logits = output.logits[0, -1, token_ids].float()
    probabilities = torch.softmax(logits, dim=-1)
    order = torch.argsort(probabilities, descending=True)
    pred_index = int(order[0].item())
    scores = {choice: float(probabilities[idx].item()) for idx, choice in enumerate("abcd")}
    scores["margin"] = float((probabilities[order[0]] - probabilities[order[1]]).item())
    return "abcd"[pred_index], "abcd"[pred_index], scores


@torch.no_grad()
def evaluate_valid_loss():
    model.eval()
    total_loss = 0.0
    total_items = 0
    for batch in valid_loss_loader:
        batch = {key: value.to(MODEL_DEVICE, non_blocking=True) for key, value in batch.items()}
        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE, enabled=True):
            output = model(**batch)
        batch_size = int(batch["input_ids"].shape[0])
        total_loss += float(output.loss.item()) * batch_size
        total_items += batch_size
    return total_loss / max(total_items, 1)


@torch.no_grad()
def evaluate_valid_accuracy(mode: str, max_samples=None):
    model.eval()
    total = len(valid_gen_ds) if max_samples is None else min(max_samples, len(valid_gen_ds))
    rows = []
    for index in tqdm(range(total), desc=f"Valid ({mode})", leave=False):
        sample = valid_gen_ds[index]
        if mode == "constrained":
            pred, raw, scores = predict_choice_constrained(sample["messages"])
        else:
            pred, raw, scores = predict_choice_generate(sample["messages"])
        rows.append({
            "id": valid_subset.iloc[index]["id"],
            "gold": sample["answer"],
            "pred": pred,
            "raw": raw,
            "valid": pred in {"a", "b", "c", "d"},
            **(scores or {}),
        })
    frame = pd.DataFrame(rows)
    accuracy = float((frame["pred"] == frame["gold"]).mean())
    invalid = int((~frame["valid"]).sum())
    return accuracy, invalid, frame


def verify_constrained_mode(sample_count=CONSTRAINED_VERIFY_SAMPLES):
    total = min(sample_count, len(valid_gen_ds))
    same = 0
    for index in tqdm(range(total), desc="Verify constrained", leave=False):
        messages = valid_gen_ds[index]["messages"]
        pred_constrained, _, _ = predict_choice_constrained(messages)
        pred_generate, _, _ = predict_choice_generate(messages)
        same += int(pred_constrained == pred_generate)
    agreement = same / max(total, 1)
    print(f"Constrained ↔ generate agreement: {same}/{total} = {agreement:.2%}")
    return agreement


def save_checkpoint(checkpoint_dir, epoch, global_step, metrics, is_best=False):
    os.makedirs(checkpoint_dir, exist_ok=True)
    model.save_pretrained(checkpoint_dir)
    processor.save_pretrained(checkpoint_dir)
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "metrics": metrics,
        "is_best": is_best,
        "selected_max_pixels": MAX_PIXELS,
    }, os.path.join(checkpoint_dir, "training_state.pt"))


# Constrained inference가 generation과 충분히 일치하는지 Base 모델에서 확인한다.
constrained_agreement = verify_constrained_mode()
resolved_eval_mode = "constrained" if constrained_agreement >= CONSTRAINED_MIN_AGREEMENT else "generate"
if VALIDATION_MODE != "auto":
    resolved_eval_mode = VALIDATION_MODE
resolved_inference_mode = resolved_eval_mode if INFERENCE_MODE == "auto" else INFERENCE_MODE
print("Validation mode:", resolved_eval_mode, "| Inference mode:", resolved_inference_mode)

# Optimizer / scheduler
trainable_params = [parameter for parameter in model.parameters() if parameter.requires_grad]
try:
    optimizer = torch.optim.AdamW(
        trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=True
    )
except TypeError:
    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_training_steps = optimizer_steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_training_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

history = []
if RUN_BASELINE_VALIDATION:
    baseline_loss = evaluate_valid_loss()
    limit = SMOKE_VAL_ACC_SAMPLES if SMOKE_TEST else None
    baseline_accuracy, baseline_invalid, baseline_frame = evaluate_valid_accuracy(resolved_eval_mode, limit)
    baseline_frame.to_csv(os.path.join(EXPERIMENT_DIR, "validation_predictions_base.csv"), index=False)
    history.append({
        "epoch": 0,
        "train_loss": np.nan,
        "valid_loss": baseline_loss,
        "valid_accuracy": baseline_accuracy,
        "invalid": baseline_invalid,
    })
    print(f"[BASE] valid_loss={baseline_loss:.4f} valid_accuracy={baseline_accuracy:.4%} invalid={baseline_invalid}")
    REPORT["baseline"] = history[-1]


Context-aware choice token IDs: {'a': 64, 'b': 65, 'c': 66, 'd': 67}


Constrained ↔ generate agreement: 64/64 = 100.00%
Validation mode: constrained | Inference mode: constrained


In [ ]:
# ============================================================
# Fine-tuning loop
# - 실제 accumulation window 크기로 loss scaling
# - gradient clipping
# - 첫 optimizer step에서 90GB hard guard
# - Best Accuracy / Best Loss를 별도로 보존
# ============================================================
start_epoch = 1
global_step = 0
best_accuracy = -float("inf")
best_val_loss = float("inf")
best_accuracy_epoch = None
best_loss_epoch = None
epochs_without_joint_improvement = 0

cleanup_cuda()
torch.cuda.reset_peak_memory_stats()

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    if hasattr(train_ds, "set_epoch"):
        train_ds.set_epoch(epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    processed_batches = 0
    step_times = []
    total_batches = len(train_loader)
    final_remainder = total_batches % GRAD_ACCUM

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", unit="batch")
    for step, batch in enumerate(progress, start=1):
        started = time.time()
        batch = {key: value.to(MODEL_DEVICE, non_blocking=True) for key, value in batch.items()}

        in_final_partial = final_remainder > 0 and step > total_batches - final_remainder
        accumulation_divisor = final_remainder if in_final_partial else GRAD_ACCUM

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE, enabled=True):
            output = model(**batch)
            loss = output.loss
            scaled_loss = loss / accumulation_divisor

        scaled_loss.backward()
        running_loss += float(loss.item())
        processed_batches += 1

        is_last = step == total_batches or (SMOKE_TEST and step >= SMOKE_MAX_TRAIN_BATCHES)
        should_step = step % GRAD_ACCUM == 0 or is_last
        grad_norm = None
        if should_step:
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            torch.cuda.synchronize()
            current_reserved = torch.cuda.max_memory_reserved() / 1024**3
            if current_reserved > MAX_VRAM_GB:
                raise RuntimeError(
                    f"VRAM hard limit exceeded: {current_reserved:.1f}GB > {MAX_VRAM_GB:.1f}GB. "
                    "런타임 재시작 후 PIXEL_CANDIDATES의 첫 값을 낮추세요."
                )

        torch.cuda.synchronize()
        step_times.append(time.time() - started)
        progress.set_postfix({
            "loss": f"{float(loss.item()):.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
            "vram": f"{torch.cuda.max_memory_reserved()/1024**3:.1f}G",
        })

        if USE_WANDB and should_step:
            wandb.log({
                "train/loss": float(loss.item()),
                "train/grad_norm": float(grad_norm) if grad_norm is not None else None,
                "learning_rate": optimizer.param_groups[0]["lr"],
                "epoch": epoch,
                "global_step": global_step,
                "vram_reserved_gb": torch.cuda.max_memory_reserved() / 1024**3,
            })

        if is_last:
            break

    train_loss = running_loss / max(processed_batches, 1)
    valid_loss = evaluate_valid_loss()
    limit = SMOKE_VAL_ACC_SAMPLES if SMOKE_TEST else None
    valid_accuracy, invalid_count, validation_frame = evaluate_valid_accuracy(resolved_eval_mode, limit)
    validation_frame.to_csv(
        os.path.join(EXPERIMENT_DIR, f"validation_predictions_epoch_{epoch:02d}.csv"), index=False
    )

    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
    metrics = {
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
        "invalid": invalid_count,
        "peak_reserved_gb": peak_reserved,
        "avg_sec_per_batch": float(np.mean(step_times[3:])) if len(step_times) > 3 else float(np.mean(step_times)),
        "selected_max_pixels": MAX_PIXELS,
    }
    history.append(metrics)

    print(
        f"\n[Epoch {epoch}] train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} "
        f"valid_accuracy={valid_accuracy:.4%} invalid={invalid_count} peak={peak_reserved:.1f}GB"
    )

    if USE_WANDB:
        wandb.log({
            "train/loss_epoch": train_loss,
            "val/loss": valid_loss,
            "val/accuracy": valid_accuracy,
            "val/invalid": invalid_count,
            "epoch": epoch,
            "global_step": global_step,
        })

    checkpoint_dir = os.path.join(EXPERIMENT_DIR, f"checkpoint-epoch-{epoch:02d}")
    if SAVE_EVERY_EPOCH:
        save_checkpoint(checkpoint_dir, epoch, global_step, metrics, is_best=False)

    accuracy_improved = valid_accuracy > best_accuracy + EARLY_STOPPING_MIN_DELTA
    loss_improved = valid_loss < best_val_loss - 1e-6

    if accuracy_improved:
        best_accuracy = valid_accuracy
        best_accuracy_epoch = epoch
        best_accuracy_dir = os.path.join(EXPERIMENT_DIR, "best_accuracy")
        if os.path.exists(best_accuracy_dir):
            shutil.rmtree(best_accuracy_dir)
        save_checkpoint(best_accuracy_dir, epoch, global_step, metrics, is_best=True)
        print("Best-accuracy checkpoint updated:", best_accuracy_dir)

    if loss_improved:
        best_val_loss = valid_loss
        best_loss_epoch = epoch
        best_loss_dir = os.path.join(EXPERIMENT_DIR, "best_loss")
        if os.path.exists(best_loss_dir):
            shutil.rmtree(best_loss_dir)
        save_checkpoint(best_loss_dir, epoch, global_step, metrics, is_best=True)
        print("Best-loss checkpoint updated:", best_loss_dir)

    if accuracy_improved or loss_improved:
        epochs_without_joint_improvement = 0
    else:
        epochs_without_joint_improvement += 1
        print(
            f"No validation improvement: patience "
            f"{epochs_without_joint_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

    # checkpoint 보관 수 제한
    checkpoints = sorted(
        os.path.join(EXPERIMENT_DIR, name)
        for name in os.listdir(EXPERIMENT_DIR)
        if name.startswith("checkpoint-epoch-")
    )
    while len(checkpoints) > SAVE_TOTAL_LIMIT:
        shutil.rmtree(checkpoints.pop(0))

    pd.DataFrame(history).to_csv(os.path.join(EXPERIMENT_DIR, "training_history.csv"), index=False)

    if epochs_without_joint_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping: accuracy와 loss가 모두 개선되지 않았습니다.")
        break

final_dir = os.path.join(EXPERIMENT_DIR, "final")
os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)

REPORT.update({
    "history": history,
    "best_accuracy": best_accuracy,
    "best_accuracy_epoch": best_accuracy_epoch,
    "best_val_loss": best_val_loss,
    "best_loss_epoch": best_loss_epoch,
    "actual_peak_reserved_gb": torch.cuda.max_memory_reserved() / 1024**3,
})

print("\nTraining finished")
print("Best accuracy:", f"{best_accuracy:.4%}", "at epoch", best_accuracy_epoch)
print("Best loss:", f"{best_val_loss:.4f}", "at epoch", best_loss_epoch)


Epoch 1/2 [train]: 100%|█████████▉| 6038/6039 [1:26:45<00:00,  1.16batch/s, loss=0.0044, lr=1.03e-05, vram=88.4G]



[Epoch 1] train_loss=0.0547 valid_loss=0.0483 valid_accuracy=94.9630% invalid=0 peak=88.4GB
Best-accuracy checkpoint updated: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_aug80_candidate_9h/best_accuracy
Best-loss checkpoint updated: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_aug80_candidate_9h/best_loss


Epoch 2/2 [train]: 100%|█████████▉| 6038/6039 [1:26:44<00:00,  1.16batch/s, loss=0.0001, lr=0.00e+00, vram=88.4G]



[Epoch 2] train_loss=0.0330 valid_loss=0.0541 valid_accuracy=95.4074% invalid=0 peak=88.4GB
Best-accuracy checkpoint updated: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_aug80_candidate_9h/best_accuracy

Training finished
Best accuracy: 95.4074% at epoch 2
Best loss: 0.0483 at epoch 1


# 자동 설정 선택과 전체 6,714장 최종 학습

후보 실험 결과를 기존 기준과 비교한다. 후보가 약 2문항 이상 개선하거나 동일 정확도에서 Loss가 낮을 때만 증강 확률 80%를 선택한다. 그렇지 않으면 기존 70%를 유지한다.

그 후 기존 LoRA를 제거하고 새 LoRA를 초기화하여 전체 Train을 총 1,510 optimizer step만 학습한다. 기존 submission 결과나 Test 예측은 학습에 사용하지 않는다.


In [ ]:
# ============================================================
# 후보 자동 선택 → LoRA 초기화 → 전체 데이터 최종 학습
# ============================================================
candidate_accuracy = float(best_accuracy)
candidate_loss = float(best_val_loss)
validation_count = len(valid_subset)
material_gain = 2.0 / max(validation_count, 1)  # 약 2문항

candidate_wins = (
    candidate_accuracy >= BASELINE_VALID_ACCURACY + material_gain - 1e-12
    or (
        abs(candidate_accuracy - BASELINE_VALID_ACCURACY) < 5e-7
        and candidate_loss < BASELINE_VALID_LOSS
    )
)
selected_augmented_probability = (
    CANDIDATE_AUGMENTED_VIEW_PROB if candidate_wins else BASELINE_AUGMENTED_VIEW_PROB
)
selected_reason = (
    "candidate_aug80_improved_validation"
    if candidate_wins
    else "baseline_aug70_retained"
)

print("=" * 72)
print("AUTO CONFIG SELECTION")
print("=" * 72)
print(f"Baseline: accuracy={BASELINE_VALID_ACCURACY:.4%}, loss={BASELINE_VALID_LOSS:.4f}")
print(f"Candidate: accuracy={candidate_accuracy:.4%}, best_loss={candidate_loss:.4f}")
print(f"Material gain threshold: {material_gain:.4%} ({validation_count} validation rows)")
print(f"Selected augmentation probability: {selected_augmented_probability:.0%}")
print("Reason:", selected_reason)

os.makedirs(FINAL_EXPERIMENT_DIR, exist_ok=True)
selection_record = {
    "baseline_valid_accuracy": BASELINE_VALID_ACCURACY,
    "baseline_valid_loss": BASELINE_VALID_LOSS,
    "candidate_valid_accuracy": candidate_accuracy,
    "candidate_best_valid_loss": candidate_loss,
    "validation_rows": validation_count,
    "material_gain": material_gain,
    "candidate_wins": candidate_wins,
    "selected_augmented_probability": selected_augmented_probability,
    "selected_reason": selected_reason,
}
with open(os.path.join(FINAL_EXPERIMENT_DIR, "auto_selection.json"), "w", encoding="utf-8") as file:
    json.dump(selection_record, file, ensure_ascii=False, indent=2)

# 후보 optimizer/scheduler를 제거하고 LoRA만 초기화한다.
# unload는 LoRA delta를 merge하지 않고 제거하므로 base weight는 원래 상태로 복구된다.
del optimizer, scheduler, trainable_params
base_model_for_final = model.unload() if isinstance(model, PeftModel) else model
del model
cleanup_cuda()

final_lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=lora_target_names,
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model_for_final, final_lora_config)
MODEL_DEVICE = next(model.parameters()).device
model.print_trainable_parameters()

# 전체 6,714장에 대응하는 증강 view만 연결한다.
if not USE_AUGMENTED_DATA:
    raise RuntimeError("최종 학습은 증강 manifest 사용을 전제로 합니다.")

full_augmented_df = pd.read_csv(AUGMENTED_CSV)
full_augmented_df["source_id"] = full_augmented_df["source_id"].astype(str)
full_augmented_df["augmentation"] = full_augmented_df["augmentation"].astype(str)
full_augmented_df["path"] = full_augmented_df["path"].map(absolute_content_path)
full_source_ids = set(train_df["id"].astype(str))
full_variants_df = full_augmented_df[
    full_augmented_df["is_augmented"].astype(bool)
    & full_augmented_df["source_id"].isin(full_source_ids)
    & full_augmented_df["augmentation"].isin(ALLOWED_AUGMENTATIONS)
].copy()

missing_variant_sources = full_source_ids - set(full_variants_df["source_id"])
if missing_variant_sources:
    print(f"[WARN] 증강 view가 없는 원본 {len(missing_variant_sources)}개는 항상 원본으로 학습됩니다.")

full_train_ds = VQAMultiViewDataset(
    train_df,
    full_variants_df,
    processor,
    augmented_probability=selected_augmented_probability,
    seed=SEED + 10_000,
)
full_loader_generator = torch.Generator().manual_seed(SEED + 10_000)
full_train_loader = DataLoader(
    full_train_ds,
    batch_size=PER_DEVICE_BATCH_SIZE,
    shuffle=True,
    generator=full_loader_generator,
    collate_fn=DataCollator(processor, True),
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

assert len(full_train_ds) == N_TRAIN_FULL == 6714, (
    len(full_train_ds), N_TRAIN_FULL
)
print("Final full-data rows:", len(full_train_ds))
print("Final variant rows available:", len(full_variants_df))
print("Final augmentation probability:", selected_augmented_probability)

final_trainable_params = [parameter for parameter in model.parameters() if parameter.requires_grad]
try:
    final_optimizer = torch.optim.AdamW(
        final_trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fused=True,
    )
except TypeError:
    final_optimizer = torch.optim.AdamW(
        final_trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

final_warmup_steps = max(1, int(FINAL_MAX_OPTIMIZER_STEPS * WARMUP_RATIO))
final_scheduler = get_linear_schedule_with_warmup(
    final_optimizer,
    num_warmup_steps=final_warmup_steps,
    num_training_steps=FINAL_MAX_OPTIMIZER_STEPS,
)

final_global_step = 0
final_processed_batches = 0
final_running_loss = 0.0
final_history = []
time_guard_triggered = False

cleanup_cuda()
torch.cuda.reset_peak_memory_stats()

for final_epoch in range(1, FINAL_MAX_EPOCHS + 1):
    full_train_ds.set_epoch(final_epoch)
    model.train()
    final_optimizer.zero_grad(set_to_none=True)
    epoch_running_loss = 0.0
    epoch_batches = 0
    total_batches = len(full_train_loader)
    final_remainder = total_batches % GRAD_ACCUM

    progress = tqdm(
        full_train_loader,
        desc=f"Final full-data epoch {final_epoch}/{FINAL_MAX_EPOCHS}",
        unit="batch",
    )
    for batch_index, batch in enumerate(progress, start=1):
        elapsed_hours = (time.time() - PIPELINE_STARTED_AT) / 3600
        if elapsed_hours >= FINAL_TRAINING_STOP_HOURS:
            time_guard_triggered = True
            print(
                f"[TIME GUARD] {elapsed_hours:.2f}h 경과. "
                "추론 시간을 확보하기 위해 최종 학습을 종료합니다."
            )
            break

        batch = {key: value.to(MODEL_DEVICE, non_blocking=True) for key, value in batch.items()}
        in_final_partial = final_remainder > 0 and batch_index > total_batches - final_remainder
        accumulation_divisor = final_remainder if in_final_partial else GRAD_ACCUM

        with torch.autocast(device_type="cuda", dtype=COMPUTE_DTYPE, enabled=True):
            output = model(**batch)
            final_loss = output.loss
            scaled_loss = final_loss / accumulation_divisor

        scaled_loss.backward()
        loss_value = float(final_loss.item())
        epoch_running_loss += loss_value
        final_running_loss += loss_value
        epoch_batches += 1
        final_processed_batches += 1

        is_epoch_end = batch_index == total_batches
        should_step = batch_index % GRAD_ACCUM == 0 or is_epoch_end
        grad_norm = None
        if should_step:
            grad_norm = torch.nn.utils.clip_grad_norm_(final_trainable_params, MAX_GRAD_NORM)
            final_optimizer.step()
            final_optimizer.zero_grad(set_to_none=True)
            final_scheduler.step()
            final_global_step += 1

            peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
            if peak_reserved > MAX_VRAM_GB:
                raise RuntimeError(
                    f"VRAM hard limit exceeded: {peak_reserved:.1f}GB > {MAX_VRAM_GB:.1f}GB"
                )

            if USE_WANDB:
                wandb.log({
                    "final_train/loss": loss_value,
                    "final_train/grad_norm": float(grad_norm),
                    "final_train/learning_rate": final_optimizer.param_groups[0]["lr"],
                    "final_train/global_step": final_global_step,
                    "final_train/epoch": final_epoch,
                })

        progress.set_postfix({
            "loss": f"{loss_value:.4f}",
            "opt_step": f"{final_global_step}/{FINAL_MAX_OPTIMIZER_STEPS}",
            "lr": f"{final_optimizer.param_groups[0]['lr']:.2e}",
            "vram": f"{torch.cuda.max_memory_reserved()/1024**3:.1f}G",
        })

        if final_global_step >= FINAL_MAX_OPTIMIZER_STEPS:
            break

    epoch_record = {
        "epoch": final_epoch,
        "optimizer_step": final_global_step,
        "processed_batches": epoch_batches,
        "mean_loss": epoch_running_loss / max(epoch_batches, 1),
        "elapsed_hours": (time.time() - PIPELINE_STARTED_AT) / 3600,
    }
    final_history.append(epoch_record)
    print("Final epoch summary:", epoch_record)

    recovery_dir = os.path.join(FINAL_EXPERIMENT_DIR, f"recovery_step_{final_global_step:04d}")
    model.save_pretrained(recovery_dir)
    processor.save_pretrained(recovery_dir)

    if final_global_step >= FINAL_MAX_OPTIMIZER_STEPS or time_guard_triggered:
        break

final_model_dir = os.path.join(FINAL_EXPERIMENT_DIR, "final_model")
os.makedirs(final_model_dir, exist_ok=True)
model.save_pretrained(final_model_dir)
processor.save_pretrained(final_model_dir)

final_training_summary = {
    "full_train_rows": len(full_train_ds),
    "selected_augmented_probability": selected_augmented_probability,
    "target_optimizer_steps": FINAL_MAX_OPTIMIZER_STEPS,
    "completed_optimizer_steps": final_global_step,
    "processed_batches": final_processed_batches,
    "mean_loss": final_running_loss / max(final_processed_batches, 1),
    "time_guard_triggered": time_guard_triggered,
    "elapsed_hours": (time.time() - PIPELINE_STARTED_AT) / 3600,
    "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1024**3,
    "history": final_history,
    "final_model_dir": final_model_dir,
}
with open(os.path.join(FINAL_EXPERIMENT_DIR, "final_training_summary.json"), "w", encoding="utf-8") as file:
    json.dump(final_training_summary, file, ensure_ascii=False, indent=2)

print("=" * 72)
print("FINAL FULL-DATA TRAINING FINISHED")
print("=" * 72)
print(json.dumps(final_training_summary, ensure_ascii=False, indent=2, default=str))


AUTO CONFIG SELECTION
Baseline: accuracy=95.1111%, loss=0.0534
Candidate: accuracy=95.4074%, best_loss=0.0483
Material gain threshold: 0.2963% (675 validation rows)
Selected augmentation probability: 80%
Reason: candidate_aug80_improved_validation
trainable params: 116,727,808 || all params: 27,473,456,368 || trainable%: 0.4249
Final full-data rows: 6714
Final variant rows available: 60334
Final augmentation probability: 0.8


Final full-data epoch 1/2: 100%|██████████| 6714/6714 [1:36:28<00:00,  1.16batch/s, loss=0.0000, opt_step=840/1510, lr=9.15e-06, vram=79.0G]


Final epoch summary: {'epoch': 1, 'optimizer_step': 840, 'processed_batches': 6714, 'mean_loss': 0.05331310015918157, 'elapsed_hours': 4.8023894222577415}


Final full-data epoch 2/2:  80%|███████▉  | 5359/6714 [1:16:52<19:26,  1.16batch/s, loss=0.0577, opt_step=1510/1510, lr=0.00e+00, vram=79.0G]


Final epoch summary: {'epoch': 2, 'optimizer_step': 1510, 'processed_batches': 5360, 'mean_loss': 0.03522670144409471, 'elapsed_hours': 6.083852644893859}
FINAL FULL-DATA TRAINING FINISHED
{
  "full_train_rows": 6714,
  "selected_augmented_probability": 0.8,
  "target_optimizer_steps": 1510,
  "completed_optimizer_steps": 1510,
  "processed_batches": 12074,
  "mean_loss": 0.04528402138554685,
  "time_guard_triggered": false,
  "elapsed_hours": 6.084240383704503,
  "peak_reserved_gb": 79.03515625,
  "history": [
    {
      "epoch": 1,
      "optimizer_step": 840,
      "processed_batches": 6714,
      "mean_loss": 0.05331310015918157,
      "elapsed_hours": 4.8023894222577415
    },
    {
      "epoch": 2,
      "optimizer_step": 1510,
      "processed_batches": 5360,
      "mean_loss": 0.03522670144409471,
      "elapsed_hours": 6.083852644893859
    }
  ],
  "final_model_dir": "/content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/final_model"
}


In [ ]:
# ============================================================
# 최종 단일 모델 Test 추론 및 제출 파일 검증
# - 다른 checkpoint 예측이나 기존 submission을 섞지 않는다.
# ============================================================
@torch.no_grad()
def run_final_single_model_inference():
    model.eval()
    predictions = []
    score_rows = []
    timings = []

    for index in tqdm(range(len(test_df)), desc="Final single-model inference", unit="sample"):
        row = test_df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        messages = build_messages(
            row["question"], row["a"], row["b"], row["c"], row["d"], image
        )
        started = time.time()
        if resolved_inference_mode == "constrained":
            pred, raw, scores = predict_choice_constrained(messages)
        else:
            pred, raw, scores = predict_choice_generate(messages)
        torch.cuda.synchronize()
        timings.append(time.time() - started)

        pred = pred if pred in {"a", "b", "c", "d"} else "a"
        predictions.append(pred)
        score_rows.append({"id": row["id"], "pred": pred, "raw": raw, **(scores or {})})

    submission = pd.DataFrame({"id": test_df["id"].values, "answer": predictions})
    submission_path = os.path.join(FINAL_EXPERIMENT_DIR, "submission_final_single_model.csv")
    scores_path = os.path.join(FINAL_EXPERIMENT_DIR, "test_scores_final_single_model.csv")
    submission.to_csv(submission_path, index=False)
    pd.DataFrame(score_rows).to_csv(scores_path, index=False)
    return submission_path, scores_path, float(np.median(timings))


final_submission_path, final_scores_path, final_median_seconds = run_final_single_model_inference()
final_submission = pd.read_csv(final_submission_path)

final_checks = {
    "전체 6,714장 최종 학습 연결": len(full_train_ds) == N_TRAIN_FULL == 6714,
    "원본별 Random View 사용": isinstance(full_train_ds, VQAMultiViewDataset),
    "목표 step 또는 시간 보호 종료": (
        final_global_step == FINAL_MAX_OPTIMIZER_STEPS or time_guard_triggered
    ),
    "VRAM 90GB 이하": final_training_summary["peak_reserved_gb"] <= MAX_VRAM_GB,
    "최종 모델 저장": os.path.exists(
        os.path.join(final_model_dir, "adapter_model.safetensors")
    ),
    "제출 행 수": len(final_submission) == len(test_df) == 6714,
    "제출 ID 고유": final_submission["id"].is_unique,
    "제출 답 형식": set(final_submission["answer"]).issubset({"a", "b", "c", "d"}),
}

print("=" * 72)
print("FINAL SINGLE-MODEL CHECKLIST")
print("=" * 72)
for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("최종 검증 실패: 위 FAIL 항목을 확인하세요.")

print("Submission:", final_submission_path)
print("Scores:", final_scores_path)
print(f"Median inference: {final_median_seconds:.3f}s/sample")
print(f"Total elapsed: {(time.time() - PIPELINE_STARTED_AT)/3600:.2f}h")


Final single-model inference: 100%|██████████| 6714/6714 [34:16<00:00,  3.27sample/s]

FINAL SINGLE-MODEL CHECKLIST
[PASS] 전체 6,714장 최종 학습 연결
[PASS] 원본별 Random View 사용
[PASS] 목표 step 또는 시간 보호 종료
[PASS] VRAM 90GB 이하
[PASS] 최종 모델 저장
[PASS] 제출 행 수
[PASS] 제출 ID 고유
[PASS] 제출 답 형식
Submission: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/submission_final_single_model.csv
Scores: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/test_scores_final_single_model.csv
Median inference: 0.286s/sample
Total elapsed: 6.66h


# 후보 검증 기록과 최종 학습 기록


In [ ]:
candidate_history_df = pd.DataFrame(history).sort_values("epoch")
final_history_df = pd.DataFrame(final_history).sort_values("epoch")

print("Candidate validation history")
display(candidate_history_df)
print("Final full-data training history")
display(final_history_df)


Candidate validation history


,epoch,train_loss,valid_loss,valid_accuracy,invalid,peak_reserved_gb,avg_sec_per_batch,selected_max_pixels
0,1,0.054716,0.048312,0.949630,0,88.437500,0.845810,802816
1,2,0.032972,0.054115,0.954074,0,88.439453,0.845478,802816


Final full-data training history


,epoch,optimizer_step,processed_batches,mean_loss,elapsed_hours
0,1,840,6714,0.053313,4.802389
1,2,1510,5360,0.035227,6.083853


# 최종 실행 리포트


In [ ]:
final_report = {
    "model_id": MODEL_ID,
    "quant_mode": QUANT_MODE,
    "selected_max_pixels": MAX_PIXELS,
    "peak_reserved_gb": final_training_summary["peak_reserved_gb"],
    "candidate": {
        "augmentation_probability": CANDIDATE_AUGMENTED_VIEW_PROB,
        "best_accuracy": candidate_accuracy,
        "best_loss": candidate_loss,
        "history": history,
    },
    "selection": selection_record,
    "final_training": final_training_summary,
    "inference_mode": resolved_inference_mode,
    "submission": final_submission_path,
    "scores": final_scores_path,
    "median_inference_seconds": final_median_seconds,
    "total_elapsed_hours": (time.time() - PIPELINE_STARTED_AT) / 3600,
}

report_path = os.path.join(FINAL_EXPERIMENT_DIR, "final_experiment_report.json")
with open(report_path, "w", encoding="utf-8") as file:
    json.dump(final_report, file, ensure_ascii=False, indent=2, default=str)

print("=" * 72)
print("Qwen3.8-27B OCR-VQA 9H AUTO-FINAL REPORT")
print("=" * 72)
print("Candidate best accuracy:", f"{candidate_accuracy:.4%}")
print("Candidate best loss:", f"{candidate_loss:.4f}")
print("Selected augmentation probability:", f"{selected_augmented_probability:.0%}")
print("Final optimizer steps:", f"{final_global_step}/{FINAL_MAX_OPTIMIZER_STEPS}")
print("Time guard triggered:", time_guard_triggered)
print("Peak VRAM:", f"{final_training_summary['peak_reserved_gb']:.1f}GB")
print("Final submission:", final_submission_path)
print("Report:", report_path)


Qwen3.8-27B OCR-VQA 9H AUTO-FINAL REPORT
Candidate best accuracy: 95.4074%
Candidate best loss: 0.0483
Selected augmentation probability: 80%
Final optimizer steps: 1510/1510
Time guard triggered: False
Peak VRAM: 79.0GB
Final submission: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/submission_final_single_model.csv
Report: /content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/final_experiment_report.json


In [ ]:
# 대회 규정 준수 확인
print("이 submission은 최종 단일 모델의 직접 추론 결과입니다.")
print("기존 submission 결합, 답안 수동 교체, 외부 예측 투표를 사용하지 않았습니다.")
print("제출 전 대회 규정과 파일 경로를 한 번 더 확인하세요:")
print(final_submission_path)


이 submission은 최종 단일 모델의 직접 추론 결과입니다.
기존 submission 결합, 답안 수동 교체, 외부 예측 투표를 사용하지 않았습니다.
제출 전 대회 규정과 파일 경로를 한 번 더 확인하세요:
/content/drive/MyDrive/vqa_checkpoints/qwen38_27b_ocr_90gb_full6714_autofinal_9h/submission_final_single_model.csv


# 실행 시 주의사항

1. Colab 런타임을 **G4 GPU**로 설정합니다.
2. 설치 셀 실행 후 `transformers`나 CUDA 패키지가 갱신됐다면 런타임을 재시작합니다.
3. Drive 마운트와 데이터 압축 해제 셀을 실행합니다.
4. 이후 셀을 순서대로 실행합니다.
5. Adaptive probe가 896→832→768→672→576으로 낮아지는 것은 정상입니다.
6. 첫 optimizer step에서 실측 VRAM이 90GB를 넘으면 학습을 즉시 중단합니다.
7. 결과물은 실험 폴더에 다음 이름으로 저장됩니다.
   - `submission_best_accuracy.csv`
   - `submission_best_loss.csv`
   - `validation_predictions_base.csv`
   - `validation_predictions_epoch_XX.csv`
   - `training_history.csv`
   - `experiment_report.json`
8. Base 정확도가 학습 후보다 높으면 추가 Epoch를 돌리지 말고 Base 또는 8B 결과를 우선합니다.
9. `best_accuracy`와 `best_loss`가 같은 Epoch이면 두 제출 파일의 예측도 동일할 수 있습니다.
